# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Setup**

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


**Create Feature Frame as per defined data contract in week 3**

In [4]:
DECISION_DATE = "2026-03-31"

In [5]:
content_state = con.sql(
    f"""
    SELECT
        content_hash_id,

        DATE_DIFF(
            'day',
            CAST(content_created_date AS DATE),
            DATE '{DECISION_DATE}'
        ) AS content_age_days

    FROM {TABLES['dim_content']}

    WHERE CAST(content_created_date AS DATE)
          <= DATE '{DECISION_DATE}'
    """
).df()

In [6]:
content_update_state = con.sql(
    f"""
    SELECT
        content_hash_id,

        CASE
            WHEN CAST(content_updated_date AS DATE)
                 <= DATE '{DECISION_DATE}'
            THEN DATE_DIFF(
                'day',
                CAST(content_updated_date AS DATE),
                DATE '{DECISION_DATE}'
            )
            ELSE NULL
        END AS days_since_last_update

    FROM {TABLES['dim_content']}

    WHERE CAST(content_created_date AS DATE)
          <= DATE '{DECISION_DATE}'
    """
).df()

In [7]:
content_state = content_state.merge(
    content_update_state,
    on="content_hash_id",
    how="left"
)

In [8]:
historical_features = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_prev30,

        SUM(gsc_clicks) AS clicks_prev30,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN
                SUM(gsc_sum_position)
                / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_prev30

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-03-30'

      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
FEATURES = [
    "imp_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_age_days",
    "days_since_last_update",
]

feature_frame = historical_features.merge(
    content_state,
    on="content_hash_id",
    how="left"
)

feature_frame = feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        *FEATURES
    ]
].copy()

In [10]:
feature_frame.head()

,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30,content_age_days,days_since_last_update
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,786.0,1.0,5.922392,47,<NA>
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,5.941176,47,<NA>
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,324.0,0.0,5.129630,47,<NA>
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,763.0,1.0,4.826999,47,<NA>
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,14.0,0.0,4.285714,47,<NA>


## 1) Signal checks and verdicts

Before building the baseline action score, I checked two decision-time signals that could support the content-refresh lane.

### Signal 1 — Staleness

**Signal:** `days_since_last_update`

This signal is linked to the real FlyRank refresh workflow, where content staleness can be a reason for a refresh flag.

The bucket results show that older pages are **less likely to have meaningful search visibility**. Pages updated within 0–90 days have 40.53% with at least 500 impressions, compared with only 2.34% for pages in the 181–365 day bucket.

**Verdict: OPPOSITE**

The data does not support the assumption that stale pages are more likely to be visibly important. Missing update dates are also very common, so missing values should remain a separate state rather than being treated as non-stale.

### Signal 2 — Search visibility / volume

**Signal:** `imp_prev30`

This represents recent search visibility and is related to the volume/visibility idea behind quick-win opportunities.

The bucket results show the opposite relationship to staleness: pages with higher impressions are progressively **less likely to be stale**. The stale rate falls from 2.41% in the 0–99 impression bucket to 0.04% in the 1000+ bucket.

**Verdict: OPPOSITE**

The data does not support using high visibility alone as evidence that a page is stale.

### Task 1 conclusion

Both tested signals produced an **OPPOSITE** verdict for the original stale-and-visible hypothesis. Therefore, the baseline should remain transparent and should not claim that these exploratory checks confirmed the hypothesis. The results instead provide a useful warning that staleness and recent visibility behave differently in this dataset.


In [11]:
import pandas as pd
import numpy as np

check_df = feature_frame.copy()

In [12]:
# ---------------------------------------------------------
# Signal 1: Staleness
# ---------------------------------------------------------
# Keep missing update dates separate.
check_df["staleness_bucket"] = pd.cut(
    check_df["days_since_last_update"],
    bins=[-np.inf, 90, 180, 365, np.inf],
    labels=["0-90d", "91-180d", "181-365d", "366d+"]
)

check_df["staleness_bucket"] = (
    check_df["staleness_bucket"]
    .astype("object")
    .where(
        check_df["days_since_last_update"].notna(),
        "missing"
    )
)

staleness_table = (
    check_df
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_imp_prev30=("imp_prev30", "mean"),
        median_imp_prev30=("imp_prev30", "median"),
        visible_pct=("imp_prev30", lambda x: (x >= 500).mean() * 100)
    )
    .reset_index()
)

print("SIGNAL 1 — STALENESS")
print(staleness_table.to_string(index=False))


# ---------------------------------------------------------
# Signal 2: Search visibility / volume
# ---------------------------------------------------------
check_df["visibility_bucket"] = pd.cut(
    check_df["imp_prev30"],
    bins=[-np.inf, 99, 499, 999, np.inf],
    labels=["0-99", "100-499", "500-999", "1000+"]
)

visibility_table = (
    check_df
    .groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_imp_prev30=("imp_prev30", "mean"),
        mean_age_days=("content_age_days", "mean"),
        stale_pct=("days_since_last_update", lambda x: (x >= 180).mean() * 100)
    )
    .reset_index()
)

print("\nSIGNAL 2 — SEARCH VISIBILITY / VOLUME")
print(visibility_table.to_string(index=False))

SIGNAL 1 — STALENESS
staleness_bucket      n  mean_imp_prev30  median_imp_prev30  visible_pct
           0-90d  26319      1248.639842              305.0    40.529655
        181-365d    256        65.941406                4.0     2.343750
         91-180d   1305       351.540230                4.0     4.061303
         missing 147325      1606.770433              157.0    34.100798

SIGNAL 2 — SEARCH VISIBILITY / VOLUME
visibility_bucket     n  mean_imp_prev30  mean_age_days  stale_pct
             0-99 74829        24.754908     179.875583   2.407201
          100-499 39411       250.362234     191.423435   0.165130
          500-999 16773       716.867943     187.590890   0.080559
            1000+ 44192      5573.692297     188.570624   0.042845


I use a simple, transparent content-refresh baseline.

Pages receive a higher score when they have meaningful recent search visibility and are older since their last known update. Pages with missing update dates are not automatically treated as stale.

Reason code: stale_and_visible

Action: REFRESH_REVIEW

The score is used only to rank pages for human review. It is a decision-support baseline, not a prediction of future decline. The rule uses only information available at the 2026-03-31 decision date.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# Task 2 — Build the ranked queue
# Decision-time features only. No future window or label-derived inputs.

import numpy as np
import pandas as pd
from pathlib import Path

queue = feature_frame.copy()

# ---------------------------------------------------------
# 1. Define transparent components
# ---------------------------------------------------------

# Visibility component:
# Pages with 500+ impressions receive visibility points.
visibility_score = np.where(
    queue["imp_prev30"] >= 500,
    1,
    0
)

# Staleness component:
# Only known update dates can be classified as stale.
stale_score = np.where(
    queue["days_since_last_update"].notna()
    & (queue["days_since_last_update"] >= 180),
    1,
    0
)

# ---------------------------------------------------------
# 2. Create baseline score
# ---------------------------------------------------------

queue["baseline_score"] = (
    visibility_score * 2
    + stale_score * 1
)

# ---------------------------------------------------------
# 3. Assign reason codes
# ---------------------------------------------------------

queue["reason_code"] = np.select(
    [
        (visibility_score == 1) & (stale_score == 1),
        (visibility_score == 1),
        (stale_score == 1),
    ],
    [
        "stale_and_visible",
        "visible_only",
        "stale_only",
    ],
    default="no_signal"
)

# ---------------------------------------------------------
# 4. Assign actions
# ---------------------------------------------------------

queue["action"] = np.select(
    [
        queue["reason_code"] == "stale_and_visible",
        queue["reason_code"] == "visible_only",
        queue["reason_code"] == "stale_only",
    ],
    [
        "REFRESH_REVIEW",
        "MONITOR",
        "REFRESH_REVIEW",
    ],
    default="NO_ACTION"
)

# ---------------------------------------------------------
# 5. Rank everything
# ---------------------------------------------------------

queue = queue.sort_values(
    ["baseline_score", "imp_prev30"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# ---------------------------------------------------------
# 6. Keep the output columns
# ---------------------------------------------------------

output = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action",
        "imp_prev30",
        "days_since_last_update",
    ]
].copy()

# ---------------------------------------------------------
# 7. Write CSV
# ---------------------------------------------------------

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(output_path, index=False)

print(f"Rows written: {len(output):,}")
print(f"Output: {output_path}")

print("\nTop 20:")
display(output.head(20))

Rows written: 175,205
Output: work/outputs/baseline_action_score.csv

Top 20:


,rank,client_hash_id,content_hash_id,baseline_score,reason_code,action,imp_prev30,days_since_last_update
0,1,client_c182d11e4862a37d,content_42ce26be1ec6be00,3,stale_and_visible,REFRESH_REVIEW,4276.0,264
1,2,client_c182d11e4862a37d,content_bea86ce3455100b0,3,stale_and_visible,REFRESH_REVIEW,3609.0,232
2,3,client_c182d11e4862a37d,content_5120dcbbb086843d,3,stale_and_visible,REFRESH_REVIEW,1426.0,247
3,4,client_65de48885f4ef01b,content_eba53d72e18a9f93,3,stale_and_visible,REFRESH_REVIEW,718.0,231
4,5,client_65de48885f4ef01b,content_c126a43258b574c3,3,stale_and_visible,REFRESH_REVIEW,591.0,231
5,6,client_c182d11e4862a37d,content_5271624ae98fff86,3,stale_and_visible,REFRESH_REVIEW,543.0,231
6,7,client_e547b89c05043229,content_eadb33b5df496f4a,2,visible_only,MONITOR,582518.0,<NA>
7,8,client_e547b89c05043229,content_ec2e0346994fb5a5,2,visible_only,MONITOR,240240.0,<NA>
8,9,client_23a62021009f63c4,content_e8a52cf3d5988c07,2,visible_only,MONITOR,236923.0,<NA>
9,10,client_23a62021009f63c4,content_44f34c0a90047651,2,visible_only,MONITOR,209710.0,<NA>


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 pages are reviewed as human-review candidates, not as confirmed refresh recommendations. Each line states the action, the reason for the ranking, and the main condition that could make the pick wrong.

1. **REFRESH_REVIEW** — 4,276 impressions and 264 days since update make this a high-visibility stale page; it could be wrong if the content was recently maintained despite the recorded update date.
2. **REFRESH_REVIEW** — 3,609 impressions and 232 days since update make this a high-visibility stale page; it could be wrong if the page remains current or the update date is incomplete.
3. **REFRESH_REVIEW** — 1,426 impressions and 247 days since update make this a visible stale page; it could be wrong if the page is still performing well without needing a refresh.
4. **REFRESH_REVIEW** — 718 impressions and 231 days since update make this a visible stale page; it could be wrong if the recorded update date does not reflect the latest meaningful change.
5. **REFRESH_REVIEW** — 591 impressions and 231 days since update make this a visible stale page; it could be wrong if the page is still relevant and up to date.
6. **REFRESH_REVIEW** — 543 impressions and 231 days since update make this a visible stale page; it could be wrong if the page's content was maintained without the update date being refreshed.
7. **MONITOR** — 582,518 impressions make this a highly visible page, but its update date is missing; it could be wrong if the missing date hides substantial staleness or recent maintenance.
8. **MONITOR** — 240,240 impressions give this page strong visibility, but no update date is available; it could be wrong if the missing date conceals a refresh opportunity.
9. **MONITOR** — 236,923 impressions make this page highly visible, but its staleness cannot be established; it could be wrong if the missing update information hides an outdated page.
10. **MONITOR** — 209,710 impressions make this page highly visible, but the update date is missing; it could be wrong if the page is actually stale.
11. **MONITOR** — 201,258 impressions make this page highly visible, but there is no known update date; it could be wrong if missing metadata hides a refresh need.
12. **MONITOR** — 197,754 impressions make this page highly visible, but staleness cannot be confirmed; it could be wrong if the page has not been maintained recently.
13. **MONITOR** — 197,506 impressions make this page highly visible, but the update date is missing; it could be wrong if the page is outdated.
14. **MONITOR** — 194,561 impressions make this page highly visible, but there is no known update date; it could be wrong if missing metadata hides staleness.
15. **MONITOR** — 188,314 impressions make this page highly visible, but its update date is unavailable; it could be wrong if the page actually needs a refresh.
16. **MONITOR** — 188,251 impressions make this page highly visible, but no update date is known; it could be wrong if the page is stale despite the missing metadata.
17. **MONITOR** — 187,427 impressions make this page highly visible, but staleness cannot be determined; it could be wrong if the page requires updating.
18. **MONITOR** — 178,025 impressions make this page highly visible, but the update date is missing; it could be wrong if the page is outdated.
19. **MONITOR** — 163,977 impressions make this page highly visible, but its update date is unavailable; it could be wrong if missing metadata hides a refresh opportunity.
20. **MONITOR** — 158,948 impressions make this page highly visible, but no update date is known; it could be wrong if the page is actually stale.

**Review conclusion:** The first six pages have both signals used by the baseline and therefore receive `REFRESH_REVIEW`. The remaining pages enter the top 20 because of high visibility alone, so they are weaker candidates and are assigned `MONITOR`. These recommendations require human validation because the baseline does not know whether the content is actually outdated or whether the recorded update metadata is complete.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The weaker picks are **ranks 7–20**. These pages receive `MONITOR` because they have high recent visibility (`imp_prev30 >= 500`), but their `days_since_last_update` value is missing. Therefore, the baseline cannot establish that they are actually stale.

For example, rank 20 has **158,948 impressions** but no recorded update date. It is included because of visibility alone, so it could be a weak pick if the page is already current or if the missing update metadata is incomplete.

The same limitation applies to ranks 7–19: their high visibility makes them worth monitoring, but the baseline has insufficient evidence to recommend a refresh.

### Leakage check

The baseline score uses only the decision-time features:

* `imp_prev30`
* `days_since_last_update`

The feature window is **2026-03-01 through 2026-03-30**, with the decision date **2026-03-31**.

No future-window metrics, decline labels, `trend_direction`, or other label-derived information were used in the score. No product flags were used either; the rule is based only on the decision-time feature frame.

Therefore, the baseline queue is **decision-time only and leakage-free based on the implemented feature construction**.


In [16]:
# Verify that the baseline score uses only decision-time inputs.

allowed_score_inputs = {
    "imp_prev30",
    "days_since_last_update"
}

# These are the actual columns used to calculate the score.
actual_score_inputs = {
    "imp_prev30",
    "days_since_last_update"
}

print("Actual score inputs:")
print(sorted(actual_score_inputs))

print("\nAllowed decision-time inputs:")
print(sorted(allowed_score_inputs))

# Check 1: only allowed inputs are used
assert actual_score_inputs.issubset(allowed_score_inputs)

# Check 2: forbidden future/label-style columns are not used
forbidden_terms = [
    "label",
    "declin",
    "trend",
    "future",
    "last30"
]

for col in actual_score_inputs:
    col_lower = col.lower()
    assert not any(term in col_lower for term in forbidden_terms), \
        f"Possible leakage column found: {col}"

# Check 3: product flags are not part of the scoring inputs
product_flag_inputs = [
    col for col in actual_score_inputs
    if "flag" in col.lower()
]

assert len(product_flag_inputs) == 0

print("\nLeakage check: PASSED")
print("Future-window / label-derived inputs used: NO")
print("Product flags used in baseline score: NO")
print("Decision date: 2026-03-31")
print("Historical feature window: 2026-03-01 to 2026-03-30")

Actual score inputs:
['days_since_last_update', 'imp_prev30']

Allowed decision-time inputs:
['days_since_last_update', 'imp_prev30']

Leakage check: PASSED
Future-window / label-derived inputs used: NO
Product flags used in baseline score: NO
Decision date: 2026-03-31
Historical feature window: 2026-03-01 to 2026-03-30


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.